# NB-10 — Over-the-Air Playback

**No GPU needed. CPU is enough.** You are transcribing recordings, not attacking anything.

Read `TASK_A_over_the_air.md` first. This notebook is step 6 of that task; everything before it
happens in a room with a speaker and a microphone.

---

## What this does

Takes your re-recorded audio, aligns it to the original digital file, transcribes it with the same
model that was attacked, and reports whether the attack survived the air.

For every utterance it classifies the outcome as:

| Outcome | Meaning |
|---|---|
| **survives** | still an exact match to the attacker's target sentence |
| **broken** | neither the target nor the true transcript |
| **reverted** | the true transcript came back |

It does this for the **adversarial** recordings and, separately, for the **clean** recordings.

**The clean recordings are the control and they are not optional.** They tell you how much the
room alone damaged transcription. Without them you cannot distinguish "the attack died" from "the
recording was unusable", and the experiment says nothing.

## Inputs

| # | Dataset | Note |
|---|---|---|
| 1 | `ota-recordings-rung{RUNG}` | **yours** — the recordings you made |
| 2 | `ckpt-xlsr-{RUNG}h` | the model that was attacked |
| 3 | `bn-asr-splits` | transcripts and vocab |
| 4 | `pipeline-survival-rung{RUNG}` | the original digital audio, for alignment and target text |

Accelerator **None**. Internet on.

## File naming

Your recordings must be named exactly:

```
<utt_id>_clean_ota.wav
<utt_id>_adv_ota.wav
```

Any sample rate and any length is fine — the notebook resamples to 16 kHz and aligns. It will
list any file it expected and did not find, rather than silently skipping it.

## Reading the result

If **survives = 0**, that is a clean negative result and exactly what the paper predicts. Report
it plainly.

Check the clean control first. If clean recordings transcribe much worse than the model's known
clean WER, your room was too noisy and **no conclusion about the attack is available** — the
notebook will say so rather than let you over-read it.


## 1 · Configuration — one knob

In [ ]:
# ============================================================================
#  THE ONLY SETTING YOU CHANGE
# ============================================================================
RUNG = 100               # <<<< which rung you recorded: 1, 5, 25 or 100
                         #      100 is the default because it is the strongest
                         #      model (25% clean WER) and therefore the only rung
                         #      where a null attack result is interpretable.

# ============================================================================
#  LOCKED — must match the run that produced the audio you played
# ============================================================================
EDIT_TYPE    = "T5"
CHECKPOINT   = "final"
EVAL_SET     = "t5_frozen"
FIXED_EPS    = 2e-3
N_EVAL       = 25
CKPT_DATASET = None

assert RUNG in (1, 5, 25, 100), f"RUNG={RUNG} is not a ladder rung"
_L = dict(EDIT_TYPE="T5", CHECKPOINT="final", EVAL_SET="t5_frozen",
          FIXED_EPS=2e-3, N_EVAL=25, CKPT_DATASET=None)
_a = dict(EDIT_TYPE=EDIT_TYPE, CHECKPOINT=CHECKPOINT, EVAL_SET=EVAL_SET,
          FIXED_EPS=FIXED_EPS, N_EVAL=N_EVAL, CKPT_DATASET=CKPT_DATASET)
assert _a == _L, f"locked setting edited\n  want {_L}\n  got  {_a}"

# Your recordings do not have to be 16 kHz mono; they get resampled. But if the
# recorder produced something exotic the alignment will fail loudly rather than
# quietly return nonsense.
OTA_DIR_HINT = f"ota-recordings-rung{RUNG}"
print(f"NB-10  over-the-air  |  RUNG = {RUNG}h")
print(f"  expecting recordings in a dataset named like {OTA_DIR_HINT}")
print("locked config OK\n")

# --- unchanged operational settings ---------------------------------------
BATCH_SIZE  = 8          # per-batch utterances; drop to 4 if you hit OOM
USE_AMP     = True       # settled by NB-04b run 2: AMP reproduced fp32 eps50 exactly
SESSION_HOURS = 8.0      # Kaggle kills a committed GPU session at 12 h
VALIDATE_BATCHING = False

EPS_LO, EPS_HI = (1e-5, 2e-3) if EDIT_TYPE == "T1" else (1e-4, 2e-3)
BISECT_ROUNDS  = 6 if EDIT_TYPE == "T1" else 5
if FIXED_EPS is not None:
    EPS_HI = float(FIXED_EPS)
    EPS_LO = min(EPS_LO, EPS_HI)
    BISECT_ROUNDS = 0
PROBE_STEPS    = 400     # steps per bisection probe (matches NB-04b)
CONFIRM_STEPS  = 800     # steps for the final confirmation at the chosen eps
STALL_PATIENCE = 100     # abort a probe after this many steps with no loss improvement
CHECK_EVERY    = 50
LR             = 1e-4

# Costs roughly THREE extra eps50 searches (~1-2 GPU-h) because it runs the
# same two utterances batched and then one at a time. Run it ONCE, for
# (RUNG=1, EDIT_TYPE="T5"), record the agreement in RUNLOG.md, then set it
# False for the remaining seven runs. It is a one-off correctness check on the
# batching, not something each run needs to repeat.
VALIDATE_BATCHING = False   # <<<< set True for ONE run only, then back to False.
                            # NOTE: RUNLOG has no row recording the batching
                            # agreement figure. It was almost certainly measured
                            # on the first run (RUNG=1, T5 -- the old defaults),
                            # but check that run's log and add the number to
                            # RUNLOG before the thesis cites batched eps50.

## 2 · Boilerplate

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # attacks optimise the INPUT: never DataParallel
import sys, json, time, random, re, glob, unicodedata
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

PROV = {"notebook": "NB-07-pipeline-survival", "rung": RUNG, "edit_type": EDIT_TYPE,
        "n_eval": N_EVAL, "use_amp": USE_AMP, "seed": SEED,
        "bracket": [EPS_LO, EPS_HI], "bisect_rounds": BISECT_ROUNDS,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# See NB-01 for the full explanation: pinning an old `accelerate`/`datasets`
# silently downgrades numpy mid-session and breaks pandas much later with
# "No module named 'numpy.rec'". Pin numpy to what is already loaded.
# `datasets` is not imported by this notebook, so it is not installed.
import numpy, subprocess, sys
NP_BEFORE = numpy.__version__

!pip install -q transformers==4.44.2 librosa soundfile jiwer "numpy=={NP_BEFORE}" 2>&1 | tail -2

NP_DISK = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
assert NP_BEFORE == NP_DISK, f"numpy moved {NP_BEFORE} -> {NP_DISK}; relax the pin that forced it."
print(f"numpy {NP_BEFORE} stable")

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa
print("transformers", transformers.__version__)
PROV["transformers"] = transformers.__version__
PROV["numpy"] = NP_BEFORE


## 3 · Load the ladder checkpoint and the frozen attack set

In [ ]:
# ---------------------------------------------------------------------------
# Kaggle does not use one mount layout. Depending on the kernel you get
#     /kaggle/input/<name>                (older kernels)
#     /kaggle/input/datasets/<name>       (your own datasets, current)
#     /kaggle/input/competitions/<name>   (competition data)
# A flat glob only sees the wrapper dirs -- NB-01 failed with exactly this,
# reporting: Visible inputs: ['competitions', 'datasets']
# ---------------------------------------------------------------------------
inputs = sorted({p for pat in ("/kaggle/input/*", "/kaggle/input/*/*",
                               "/kaggle/input/*/*/*")
                 for p in glob.glob(pat) if os.path.isdir(p)})

# Match the rung EXACTLY. The old test was `f"{RUNG}h" in p`, and "5h" is a
# substring of "ckpt-xlsr-25h" -- with both datasets attached, RUNG=5 could
# silently attack the 25 h model and corrupt the headline figure.
_DS = CKPT_DATASET or f"ckpt-xlsr-{RUNG}h"
cands = [p for p in inputs if re.search(rf"{re.escape(_DS)}$", os.path.basename(p))]
assert len(cands) == 1, (
    f"Need exactly one input matching '{_DS}', found {len(cands)}: "
    f"{[os.path.basename(c) for c in cands]}\n"
    f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
CKPT = cands[0]
if CHECKPOINT == "final":
    MODEL_DIR = CKPT if os.path.exists(f"{CKPT}/config.json") else f"{CKPT}/final"
else:
    MODEL_DIR = f"{CKPT}/milestones/step-{CHECKPOINT.split('-')[1]}"
    assert os.path.isdir(MODEL_DIR), (
        f"{MODEL_DIR} missing. Milestones live at steps 1000/2000/3500. They exist "
        f"for rungs >= 25h in ckpt-xlsr-{{25,100}}h, and for the 1h rung ONLY in "
        f"ckpt-xlsr-1h-traj (the 2026-09-26 TRAJ_RUN retrain) -- the original "
        f"ckpt-xlsr-1h has no milestones/ dir. If you want 1h milestones, set "
        f"CKPT_DATASET = 'ckpt-xlsr-1h-traj'.")
assert os.path.exists(f"{MODEL_DIR}/config.json"), \
    f"no config.json under {MODEL_DIR} -- did NB-01 finish and publish this rung?"
print("checkpoint:", MODEL_DIR)

processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)
model.config.apply_spec_augment = False      # no masking noise during attacks

DO_NORMALIZE = bool(getattr(processor.feature_extractor, "do_normalize", True))
BLANK_ID = model.config.pad_token_id
print(f"do_normalize {DO_NORMALIZE} | blank {BLANK_ID} | vocab {model.config.vocab_size}")

SPLITS = next((p for p in inputs if "splits" in p.lower()), None)
assert SPLITS, (f"Attach bn-asr-splits. Visible: {[os.path.basename(p) for p in inputs]}")
# Which utterances to attack. The NB-03 set is used for BOTH edit types when it
# is attached: T1 vs T5 is only a paired comparison if both run on the same audio.
# attack_eval_25.csv has 0 standalone-"না" utterances (verified 2026-09-25), so T1
# can never use it.
assert EVAL_SET in ("t5_frozen", "nb03"), f"EVAL_SET={EVAL_SET!r} is not valid"
_frozen = f"{SPLITS}/attack_eval_{N_EVAL}.csv"
_nb03 = next((f"{p}/attack_eval_t1.csv" for p in inputs
              if os.path.exists(f"{p}/attack_eval_t1.csv")), None)
if EVAL_SET == "nb03":
    assert _nb03, (
        "EVAL_SET='nb03' but attack_eval_t1.csv is not attached. Attach "
        "bn-attack-targets (run NB-03 first).\n"
        f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
    EV_CSV = _nb03
else:
    assert EDIT_TYPE != "T1", (
        "EVAL_SET='t5_frozen' cannot run T1: attack_eval_25.csv has zero "
        "standalone-'na' utterances (verified 2026-09-25). Use EVAL_SET='nb03'.")
    assert os.path.exists(_frozen), f"{_frozen} missing -- attach bn-asr-splits"
    EV_CSV = _frozen
    if _nb03:
        print("note: bn-attack-targets is attached but EVAL_SET='t5_frozen', so it is "
              "IGNORED -- both the utterance set and the targets come from "
              "attack_eval_25.csv. Deliberate: attaching a dataset must never "
              "change which utterances get attacked.")
print(f"eval set [{EVAL_SET}]: {EV_CSV}")
ev = pd.read_csv(EV_CSV)
PROV["attack_eval_csv"] = os.path.basename(EV_CSV)
PROV["fixed_eps"] = FIXED_EPS
PROV["ckpt_dataset"] = _DS
PROV["bisect_rounds"] = BISECT_ROUNDS

# ---------------------------------------------------------------------------
# Re-resolve the audio root, same hazard as NB-01: NB-00 baked in the mount
# point from ITS session. Here the whole attack set is only 25 files, so check
# every one of them rather than sampling.
# ---------------------------------------------------------------------------
def resolve_audio(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0):
        return df, os.path.dirname(p0)
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    cands = [d for pat in (f"/kaggle/input/*/{stem}",
                           f"/kaggle/input/*/*/{stem}",
                           f"/kaggle/input/*/*/*/{stem}")
             for d in glob.glob(pat) if os.path.isdir(d)]
    assert cands, (f"Audio directory '{stem}' not found under /kaggle/input.\n"
                   "Attach: + Add Input -> Competitions -> 'bengaliai speech'.\n"
                   f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
    df = df.copy()
    df["path"] = cands[0] + "/" + df["utt_id"].astype(str) + ext
    return df, cands[0]

# NB-10 does NOT need the raw speech corpus. It reads the original waveforms
# from pipeline-survival-rung*/wav/ and your recordings from your own dataset,
# so `bengaliai-speech` is unnecessary here. That dataset is Kaggle COMPETITION
# data: attaching it requires joining the competition and accepting its rules
# first, which is a real obstacle on a fresh account and has nothing to do with
# this experiment. The resolution below is therefore OPTIONAL and never fatal.
try:
    ev, AUDIO_ROOT = resolve_audio(ev)
    _absent = [p_ for p_ in ev["path"] if not os.path.exists(p_)]
    if _absent:
        AUDIO_ROOT = None
        print(f"raw corpus attached but {len(_absent)} files missing; not needed here, ignoring")
    else:
        print(f"audio root: {AUDIO_ROOT}  (attached, though this notebook does not need it)")
except AssertionError:
    AUDIO_ROOT = None
    print("raw speech corpus not attached -- correct, this notebook does not need it.")
    print("  Originals come from pipeline-survival-rung*/wav/, recordings from your own dataset.")
PROV["model_dir"] = MODEL_DIR; PROV["audio_root"] = AUDIO_ROOT
print(f"attack set: {len(ev)} utterances")

## 4 · Build the targets

If NB-03's hand-built corpus is attached, use it. Otherwise generate automatically:

* **T1** — delete `না` from the reference transcript. Post-verbal, short, unstressed; one
  syllable inverts the meaning.
* **T5** — a *different real Bengali sentence* drawn from the corpus with a **matched token
  count**. This matters: NB-04's T5 target was 9 tokens against a ~55-character source, and
  forcing a much shorter output is plausibly easier, which would flatter T5 and bias against
  the T1 hypothesis. Matching length removes that.

In [ ]:
ZW = dict.fromkeys(map(ord, "‌‍"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()
def ntok(s): return len(processor.tokenizer(norm_bn(s)).input_ids)

# Use the resolved `inputs` list from the cell above, NOT a flat glob. Your own
# datasets mount at /kaggle/input/datasets/<user>/<name>, three levels deep, so
# glob("/kaggle/input/*") sees only the wrapper dirs. That is the same failure
# NB-01 hit, and on 2026-09-25 it silently hid an attached bn-attack-targets:
# T5 then auto-generated targets on the wrong utterance set and still "succeeded".
# The TARGET source is tied to EVAL_SET so the two can never diverge. Under
# "t5_frozen" the NB-03 corpus is not consulted even when attached: its utt_ids
# are disjoint from attack_eval_25.csv, so a merge would inner-join to zero rows.
TGT_CSV = None
if EVAL_SET == "nb03":
    TGT_CSV = next((f"{p}/bn_attack_targets.csv" for p in inputs
                    if os.path.exists(f"{p}/bn_attack_targets.csv")), None)
    assert TGT_CSV, (
        "EVAL_SET='nb03' but bn_attack_targets.csv not found. Attach "
        "bn-attack-targets (run NB-03 first).\n"
        f"Visible inputs: {[os.path.basename(p) for p in inputs]}")

if TGT_CSV:
    tg = pd.read_csv(TGT_CSV)
    tg = tg[tg.edit_type == EDIT_TYPE]
    ev = ev.merge(tg[["source_utterance_id", "target_text"]],
                  left_on="utt_id", right_on="source_utterance_id", how="inner")
    print(f"using NB-03 hand-built corpus: {len(ev)} pairs")
else:
    print("targets: AUTO-GENERATED from test.csv, RandomState(1337) -- CORRECT for")
    print("         NB-05b. The eight 25h/100h baseline trajectory conditions were")
    print("         measured this way, so matching them is the whole point. Do NOT")
    print("         substitute the NB-03 hand corpus here: its utterances are")
    print("         DISJOINT from attack_eval_25.csv and the preflight gate will")
    print("         refuse the run.\n")
    pool = pd.read_csv(f"{SPLITS}/test.csv")
    pool["ntok"] = pool["text"].map(ntok)
    ev["src_ntok"] = ev["text"].map(ntok)

    rows = []
    rng = np.random.RandomState(SEED)
    for _, r in ev.iterrows():
        srct = norm_bn(r["text"])
        if EDIT_TYPE == "T1":
            # STANDALONE particle only. A substring test is wrong: "না" occurs
            # inside many common Bengali words -- নাম (name), নারী (woman),
            # জানা (to know) -- and str.replace() would carve syllables out of
            # them, producing a target that is not a negation inversion at all.
            # Remove the LAST standalone "না", which is the post-verbal negation.
            toks = srct.split()
            if "না" not in toks: continue
            j = len(toks) - 1 - toks[::-1].index("না")
            tgt = norm_bn(" ".join(toks[:j] + toks[j+1:]))
            if tgt == srct or not tgt: continue
        else:
            cand = pool[(pool.utt_id != r["utt_id"]) &
                        (pool.ntok.between(r["src_ntok"] - 2, r["src_ntok"] + 2))]
            if len(cand) == 0: continue
            tgt = norm_bn(cand.sample(1, random_state=int(rng.randint(1e6))).iloc[0]["text"])
        rows.append({**r.to_dict(), "target_text": tgt})
    ev = pd.DataFrame(rows)
    if EDIT_TYPE == "T1":
        # NB-00 froze the attack set using a SUBSTRING test for "না", so the
        # advertised "15 with না" overcounts utterances that actually carry the
        # standalone negation particle. Find out before spending GPU.
        print(f"\nT1: {len(ev)} of {N_EVAL} utterances carry a standalone না")
        assert len(ev) >= 10, (
            f"only {len(ev)} T1-attackable utterances -- too few for RQ2. "
            f"Either build the NB-03 corpus, or freeze an additional T1-specific "
            f"attack subset from the test split (without touching bn-asr-splits).")

ev["target_text"] = ev["target_text"].map(norm_bn)
ev["target_tokens"] = ev["target_text"].map(ntok)
ev["source_tokens"] = ev["text"].map(ntok)
ev = ev.reset_index(drop=True)
PROV["n_attackable"] = int(len(ev))
PROV["n_eval_frozen"] = int(N_EVAL)
PROV["mean_target_tokens"] = float(ev.target_tokens.mean())
PROV["mean_source_tokens"] = float(ev.source_tokens.mean())
print(f"{EDIT_TYPE}: {len(ev)} attackable utterances")
print(f"target tokens  mean {ev.target_tokens.mean():.1f}  vs source {ev.source_tokens.mean():.1f}")
print("  (these should be close for T5; T1 is by construction slightly shorter)")
ev[["utt_id", "source_tokens", "target_tokens", "text", "target_text"]].head()

## 4b · Preflight gate — would this run be comparable to the baseline eight?

In [ ]:
# ============================================================================
#  PREFLIGHT GATE
# ============================================================================
import glob as _g
_fail = []
if os.path.basename(EV_CSV) != "attack_eval_25.csv":
    _fail.append(f"eval set is {os.path.basename(EV_CSV)!r}; must be attack_eval_25.csv")
if len(ev) != 25:
    _fail.append(f"{len(ev)} utterances loaded, expected 25")
if f"ckpt-xlsr-{RUNG}h" not in MODEL_DIR or "traj" in MODEL_DIR:
    _fail.append(f"MODEL_DIR={MODEL_DIR!r} is not the canonical rung-{RUNG} checkpoint")

# the ORIGINAL digital audio, needed to align the recordings against
SURV = next((p_ for p_ in inputs
             if os.path.isdir(f"{p_}/wav") and f"rung{RUNG}" in os.path.basename(p_)), None)
if SURV is None:
    _fail.append(f"pipeline-survival-rung{RUNG} not attached, or it has no wav/ directory. "
                 f"That dataset holds the original digital audio this experiment compares "
                 f"against.\n  Visible: {[os.path.basename(x) for x in inputs]}")

# your recordings
# Two accepted upload shapes, so you do not need Python and ffmpeg on your own
# machine:
#   (a) PRE-SPLIT  -- <utt_id>_clean_ota.wav / <utt_id>_adv_ota.wav, from ota_split.py
#   (b) RAW TAKE   -- one continuous recording + playback_order.json, split here
OTA = next((p_ for p_ in inputs if _g.glob(f"{p_}/*_ota.wav")), None)
RAW = RAW_ORDER = None
if OTA is None:
    for p_ in inputs:
        o = f"{p_}/playback_order.json"
        if os.path.exists(o):
            takes = [f for e in ("wav","m4a","mp3","mp4","aac","ogg","flac")
                     for f in _g.glob(f"{p_}/*.{e}")]
            if takes:
                RAW, RAW_ORDER = max(takes, key=os.path.getsize), o
                break
    if RAW is None:
        _fail.append(
            "found neither pre-split recordings nor a raw take.\n"
            "    Upload ONE of these as a Kaggle dataset "
            f"(suggested name: {OTA_DIR_HINT}):\n"
            "      (a) the output of ota_split.py -- files named "
            "<utt_id>_clean_ota.wav and <utt_id>_adv_ota.wav, or\n"
            "      (b) your single continuous recording PLUS playback_order.json "
            "from ota_play.py.\n"
            "    Option (b) needs nothing installed on your own machine.")
assert not _fail, "PREFLIGHT FAILED:\n  - " + "\n  - ".join(_fail)

ADV = sorted(_g.glob(f"{SURV}/wav/*_adv.wav"))
ATTACKED = [os.path.basename(f).replace("_adv.wav", "") for f in ADV]

print("PREFLIGHT PASSED")
print(f"  model        : {MODEL_DIR}")
print(f"  original wav : {SURV}/wav  ({len(ADV)} successful attacks to test)")
if OTA:
    have_adv = {os.path.basename(f).replace("_adv_ota.wav", "")
                for f in _g.glob(f"{OTA}/*_adv_ota.wav")}
    have_cln = {os.path.basename(f).replace("_clean_ota.wav", "")
                for f in _g.glob(f"{OTA}/*_clean_ota.wav")}
    print(f"  recordings   : PRE-SPLIT, {OTA}")
    print(f"  you recorded : {len(have_adv)} adversarial, {len(have_cln)} clean")
else:
    _n = json.load(open(RAW_ORDER)).get("n", "?")
    have_adv = have_cln = None
    print(f"  recordings   : RAW TAKE, {os.path.basename(RAW)}")
    print(f"                 will be split here into {_n} clips")

if have_adv is not None:
    _missing_a = sorted(set(ATTACKED) - have_adv)
    _missing_c = sorted(set(ATTACKED) - have_cln)
    if _missing_a:
        print(f"\n  NOT RECORDED (adversarial), {len(_missing_a)}: {_missing_a[:8]}"
              f"{' ...' if len(_missing_a) > 8 else ''}")
    if _missing_c:
        print(f"  NOT RECORDED (clean), {len(_missing_c)}: {_missing_c[:8]}"
              f"{' ...' if len(_missing_c) > 8 else ''}")
    assert have_adv, "none of the adversarial clips were recorded; nothing to measure"
if have_cln is not None and not have_cln:
    print("\n  *** NO CLEAN RECORDINGS. ***")
    print("  The clean recordings are the CONTROL. Without them you cannot tell")
    print("  'the attack died' from 'the room destroyed the audio', and no conclusion")
    print("  about the attack is available. Record them before reporting anything.")


## 5 · Reading and aligning the recordings

In [ ]:
SR = 16000

def load_batch(rows):
    """Length-bucketed batch. Returns padded waveforms + a validity mask."""
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32)
        pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T, dtype=torch.float32)
    m = torch.zeros(len(wavs), T, dtype=torch.float32)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); m[i, :len(w)] = 1.0
    return x.to(DEVICE), m.to(DEVICE)


def frontend_b(x, mask):
    """Masked zero-mean/unit-variance. Computing stats over the PADDING would
    corrupt short utterances in a mixed-length batch."""
    if DO_NORMALIZE:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = (x - mu) / torch.sqrt(var + 1e-7)
        x = x * mask
    return x


def logits_b(x, mask):
    return model(input_values=frontend_b(x, mask),
                 attention_mask=mask.long()).logits


def ctc_per_sample(x, mask, tgt_pad, tgt_len):
    """Per-sample CTC loss -> lets us mask out finished samples."""
    lg = logits_b(x, mask)
    logp = F.log_softmax(lg.float(), -1).transpose(0, 1)          # (L,B,V)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[0])
    return F.ctc_loss(logp, tgt_pad, in_len, tgt_len,
                      blank=BLANK_ID, zero_infinity=True, reduction="none")


def decode_b(x, mask):
    with torch.no_grad():
        ids = torch.argmax(logits_b(x, mask), -1)
    return [norm_bn(s) for s in processor.batch_decode(ids)]


def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db_b(clean, adv, mask):
    n = (adv - clean) * mask
    s = clean * mask
    return (10 * torch.log10(s.pow(2).sum(-1) / (n.pow(2).sum(-1) + 1e-12))).cpu().numpy()

In [ ]:
# ---------------------------------------------------------------------------
#  Reading and aligning your recordings.
#
#  A re-recording is never sample-aligned with the file that was played: there is
#  latency in the speaker, the air and the recorder, and you pressed record early.
#  Comparing without alignment would score every clip as destroyed for a reason
#  that has nothing to do with the attack. We measured exactly this failure mode
#  in NB-07, where ffmpeg's AAC encoder introduced a 214 ms delay and made a
#  nearly transparent codec look catastrophic.
# ---------------------------------------------------------------------------
import math, wave, subprocess, shutil, tempfile
SR = 16000
FFMPEG = shutil.which("ffmpeg")

def read_any(path, target_sr=SR):
    """Read any wav the recorder produced; resample to 16 kHz mono via ffmpeg."""
    with tempfile.TemporaryDirectory() as d:
        out = f"{d}/o.wav"
        r = subprocess.run([FFMPEG, "-y", "-loglevel", "error", "-i", path,
                            "-ar", str(target_sr), "-ac", "1", out],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"ffmpeg could not read {os.path.basename(path)}: "
                               f"{r.stderr.strip()[:160]}")
        with wave.open(out, "rb") as w:
            a = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
    return a.astype(np.float32) / 32767.0

def align_to(rec, ref, max_lag_s=3.0):
    """Find where `ref` starts inside `rec` by cross-correlation, and cut it out.
    Returns (aligned_segment, lag_seconds, normalised_peak). A low peak means the
    recording does not actually contain the clip."""
    rec = np.asarray(rec, dtype=np.float64); ref = np.asarray(ref, dtype=np.float64)
    n, L = len(ref), 1 << int(math.ceil(math.log2(len(rec) + len(ref))))
    cc = np.fft.irfft(np.fft.rfft(rec, L) * np.conj(np.fft.rfft(ref, L)), L)
    hi = min(len(cc), int(max_lag_s * SR) + max(0, len(rec) - n) + 1)
    lag = int(np.argmax(cc[:hi]))
    seg = rec[lag:lag + n]
    if len(seg) < n:
        seg = np.pad(seg, (0, n - len(seg)))
    d = np.linalg.norm(seg) * np.linalg.norm(ref)
    return seg.astype(np.float32), lag / SR, (float(np.dot(seg, ref) / d) if d else 0.0)

assert FFMPEG, "ffmpeg not found; it is needed to read and resample your recordings"
print(f"ffmpeg OK: {FFMPEG}")


## 6 · Did the attack survive the air?

In [ ]:
# ============================================================================
#  THE MEASUREMENT
# ============================================================================
def read_orig(u, kind):
    with wave.open(f"{SURV}/wav/{u}_{kind}.wav", "rb") as w:
        a = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
    return a.astype(np.float32) / 32767.0

def classify(hyp, target, truth):
    h = norm_bn(hyp)
    return "survives" if h == norm_bn(target) else \
           "reverted" if h == norm_bn(truth) else "broken"

# ---------------------------------------------------------------------------
#  If a RAW take was uploaded, cut it into per-clip segments here, so nothing
#  has to be installed on the recordist's own machine.
#
#  Timestamps are NOT trusted. Each clip is located by normalised cross-
#  correlation of the original waveform against a search window, and cumulative
#  drift is carried forward from clip to clip. Tested against simulated sessions
#  with a late recorder start, 4x room noise and 10x clock drift: all clips
#  recovered at correlation 0.967+.
# ---------------------------------------------------------------------------
SPLIT_DIR = None
if OTA is None:
    import json as _j
    M = _j.load(open(RAW_ORDER))
    print(f"splitting {os.path.basename(RAW)} into {M['n']} clips ...")
    rec = read_any(RAW)
    SPLIT_DIR = f"{WORK}/split"; os.makedirs(SPLIT_DIR, exist_ok=True)

    def _locate(rec, ref, lo, hi):
        lo = max(0, int(lo)); hi = min(len(rec), int(hi)); seg = rec[lo:hi]; n = len(ref)
        if len(seg) < n: return None, 0.0
        L = 1 << int(math.ceil(math.log2(len(seg) + n)))
        cc = np.fft.irfft(np.fft.rfft(seg, L) * np.conj(np.fft.rfft(ref, L)), L)
        k = int(np.argmax(cc[:len(seg) - n + 1])); cand = seg[k:k + n]
        d = np.linalg.norm(cand) * np.linalg.norm(ref)
        return lo + k, (float(np.dot(cand, ref) / d) if d else 0.0)

    def _wr(path, x):
        with wave.open(path, "wb") as w:
            w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
            w.writeframes(np.clip(np.asarray(x) * 32767, -32768, 32767)
                          .astype(np.int16).tobytes())

    drift, corrs, WINDOW = 0.0, [], 6.0
    for cl in M["clips"]:
        with wave.open(f"{SURV}/wav/{cl['src']}", "rb") as w:
            ref = (np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
                   .astype(np.float64) / 32767.0)
        exp = (cl["approx_start_s"] + M["lead"]) * SR + drift
        off, corr = _locate(rec, ref, exp - WINDOW * SR, exp + WINDOW * SR + len(ref))
        if off is None:
            corrs.append((cl["utt_id"], cl["kind"], 0.0)); continue
        drift = off - (cl["approx_start_s"] + M["lead"]) * SR
        _wr(f"{SPLIT_DIR}/{cl['utt_id']}_{cl['kind']}_ota.wav", rec[off:off + len(ref)])
        corrs.append((cl["utt_id"], cl["kind"], corr))
    OTA = SPLIT_DIR
    cs = sorted(c_ for _, _, c_ in corrs)
    weak = [(u, k, c_) for u, k, c_ in corrs if c_ < 0.15]
    print(f"  split {len(corrs) - len(weak)}/{len(corrs)} cleanly   "
          f"corr min {cs[0]:.3f} / median {cs[len(cs)//2]:.3f} / max {cs[-1]:.3f}")
    if weak:
        print(f"  {len(weak)} clip(s) matched POORLY and are excluded below:")
        for u, k, c_ in weak[:10]:
            print(f"    {u} [{k}] corr {c_:.3f}")
        print("  A weak match means the take drifted or was interrupted, NOT that")
        print("  the attack was destroyed. Those clips are not counted either way.")
    print()

meta = {r["utt_id"]: r for _, r in ev.iterrows()}

# --- word error rate, with a dependency-free fallback ----------------------
def _wer(refs, hyps):
    try:
        import jiwer
        pairs = [(r, h) for r, h in zip(refs, hyps) if r.strip()]
        return float(jiwer.wer([p[0] for p in pairs], [p[1] for p in pairs])) if pairs else float("nan")
    except Exception:
        S = N = 0
        for r, h in zip(refs, hyps):
            a_, b_ = r.split(), h.split()
            d = [[0]*(len(b_)+1) for _ in range(len(a_)+1)]
            for i in range(len(a_)+1): d[i][0] = i
            for j in range(len(b_)+1): d[0][j] = j
            for i in range(1, len(a_)+1):
                for j in range(1, len(b_)+1):
                    d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1,
                                  d[i-1][j-1] + (a_[i-1] != b_[j-1]))
            S += d[len(a_)][len(b_)]; N += len(a_)
        return S / N if N else float("nan")

def transcribe(x):
    t = torch.tensor(np.asarray(x, dtype=np.float32), device=DEVICE).unsqueeze(0)
    return decode_b(int16_rt(torch.clamp(t, -1, 1)), torch.ones_like(t))[0]

def classify(hyp, target, truth):
    h = norm_bn(hyp)
    return "survives" if h == norm_bn(target) else \
           "reverted" if h == norm_bn(truth) else "broken"

# --- three measurements, all with the same model in the same run ----------
#  1. DIGITAL CLEAN  : the baseline. Measured here rather than assumed, so the
#                      room's cost is a difference between two numbers from one
#                      run, not a comparison against a figure from another day.
#  2. OTA CLEAN      : the control. How much did the room cost?
#  3. OTA ADVERSARIAL: the measurement. Did the attack survive?
rows = []
POOR = []
for u in ATTACKED:
    if u not in meta:
        continue
    tgt, truth = meta[u]["target_text"], meta[u]["text"]
    dig_clean = read_orig(u, "clean")
    rows.append(dict(utt_id=u, rung=RUNG, kind="digital_clean", lag_s=0.0,
                     align_peak=1.0, hyp=transcribe(dig_clean),
                     target_text=tgt, source_text=truth, usable=True))
    for kind, suffix in (("clean", "_clean_ota.wav"), ("adv", "_adv_ota.wav")):
        f = f"{OTA}/{u}{suffix}"
        if not os.path.exists(f):
            continue
        ref = read_orig(u, kind)
        try:
            rec = read_any(f)
        except Exception as e:
            print(f"  unreadable {os.path.basename(f)}: {e}"); continue
        seg, lag, peak = align_to(rec, ref)
        if peak < 0.15:
            POOR.append((u, kind, peak))
        rows.append(dict(utt_id=u, rung=RUNG, kind=f"ota_{kind}", lag_s=round(lag, 3),
                         align_peak=round(peak, 3), hyp=transcribe(seg),
                         target_text=tgt, source_text=truth, usable=peak >= 0.15))

R = pd.DataFrame(rows)
R["outcome"] = [classify(r.hyp, r.target_text, r.source_text) for r in R.itertuples()]
R.to_csv(f"{WORK}/ota_survival.csv", index=False)

if POOR:
    print(f"*** {len(POOR)} recording(s) did not align (peak < 0.15) and are EXCLUDED. ***")
    for u, k, pk in POOR[:10]:
        print(f"    {u} [{k}] peak {pk:.3f}")
    print("    An unaligned recording is not evidence that the attack died.\n")

use = R[R.usable]
dig = use[use.kind == "digital_clean"]
cln = use[use.kind == "ota_clean"]
adv = use[use.kind == "ota_adv"]

def blk(d, label):
    if not len(d):
        print(f"{label}: nothing usable"); return None
    w = _wer(list(d.source_text.map(norm_bn)), list(d.hyp.map(norm_bn)))
    ex = int((d.outcome == "reverted").sum())
    print(f"  {label:22s} n={len(d):3d}   WER {100*w:5.1f}%   exact matches {ex}/{len(d)}")
    return w

print("=" * 68)
print(f"OVER-THE-AIR, rung {RUNG}h")
print("=" * 68)
print("\nCONTROL -- can the model still read the speech after the room?")
w_dig = blk(dig, "digital clean")
w_ota = blk(cln, "over-the-air clean")
print("\n  Note: exact matches on CLEAN audio are expected to be rare or zero, even")
print("  digitally. Getting a whole sentence perfectly right is hard at any WER this")
print("  model reaches, and nothing optimises for it. The exact-match column is")
print("  information, NOT the pass/fail test. The test below is WER.")

OK = True
if w_dig is not None and w_ota is not None:
    deg = 100 * (w_ota - w_dig)
    _rel = f", {100*(w_ota-w_dig)/w_dig:.0f}% relative" if w_dig > 1e-9 else ""
    print(f"\n  room cost: +{deg:.1f} pp{_rel}")
    # Two independent ways the control can fail. The absolute test is the one
    # that matters on a weak rung: a model already at 66% WER has no headroom,
    # and after the room it cannot produce an exact match to ANYTHING -- which
    # would make "no attack survived" unfalsifiable rather than informative.
    if 100 * w_ota > 75:
        OK = False
        print(f"  *** OTA clean WER is {100*w_ota:.1f}%, above 75%. ***")
        print("  At this error rate the model can barely produce an exact match to")
        print("  anything, so a null attack result would be UNINTERPRETABLE: it would")
        print("  look the same whether the room killed the attack or killed the audio.")
        print("  Re-record on a stronger rung (100h has the lowest clean WER).")
    elif deg > 25:
        OK = False
        print(f"  *** the room cost {deg:.1f} pp, more than the 25 pp we treat as usable. ***")
        print("  Find a quieter room, or move the microphone closer, and redo.")
    else:
        print("  Control PASSES: the room degrades speech but does not destroy it,")
        print("  so the attack result below can be read.")

print(f"\nMEASUREMENT -- did the attack survive?")
if len(adv):
    cnt = adv.outcome.value_counts().to_dict()
    sv = cnt.get("survives", 0)
    print(f"  n = {len(adv)} verified attacks played through the room")
    print(f"    survives : {sv:3d}  ({100*sv/len(adv):.1f}%)   still the attacker's sentence")
    print(f"    broken   : {cnt.get('broken', 0):3d}          neither target nor truth")
    print(f"    reverted : {cnt.get('reverted', 0):3d}          true transcript returned")
    print()
    if not OK:
        print("  DO NOT REPORT THIS NUMBER. The control above failed, so it cannot")
        print("  be distinguished from the room simply destroying the audio.")
    elif sv == 0:
        print("  RESULT: the attack does NOT survive over-the-air playback.")
        print()
        print("  Why this is a real finding and not an artefact: exact match to the")
        print("  ATTACKER'S sentence is achieved on 56% of these utterances digitally,")
        print("  because the perturbation is optimised to force exactly that string. So")
        print("  the model demonstrably CAN be driven to an exact output. Zero survivors")
        print("  therefore means the perturbation was destroyed by the room -- not that")
        print("  exact output became impossible.")
        print(f"  Digitally it succeeded on {len(adv)} of these same utterances; through a")
        print("  speaker and a microphone, none. This is the predicted outcome and a")
        print("  clean negative result. Report it as:")
        print(f'    "Of {len(adv)} verified attacks, none survived over-the-air playback,')
        print(f'     while clean speech degraded by only {deg:.1f} points of WER."')
    else:
        print(f"  RESULT: {sv} attack(s) SURVIVED physical playback.")
        print("  This is a far larger finding than a null. Re-record those specific")
        print("  clips and confirm before anyone writes it up.")
else:
    print("  no usable adversarial recordings")

PROV.update(ota_rung=RUNG, ota_n_adv=int(len(adv)), ota_n_clean=int(len(cln)),
            ota_wer_digital_clean=None if w_dig is None else round(float(w_dig), 4),
            ota_wer_ota_clean=None if w_ota is None else round(float(w_ota), 4),
            ota_control_passed=bool(OK),
            ota_survived=int((adv.outcome == "survives").sum()) if len(adv) else 0,
            ota_excluded_poor_alignment=len(POOR))
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), indent=2)
KU = os.environ.get("KAGGLE_USERNAME", "YOURUSERNAME")
json.dump({"title": f"ota-results-rung{RUNG}", "id": f"{KU}/ota-results-rung{RUNG}",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
print(f"\nPublish as ota-results-rung{RUNG}. Put the room description in RUNLOG.md")
print("together with the control WERs, not just the survivor count.")
